# Week 6 Lab：Discrete diffusion 與 factorization error

## 學習目標
- 在偶數 parity 資料上實作 absorbing-mask forward process。
- 用完整狀態列舉精確算 $p(x_0^i\mid x_t)$。
- 分開觀察 exact joint reverse 與 factorized marginal reverse 的誤差。

> **誠實註記**：條件機率由 128 個合法狀態直接列舉，是 oracle toy；沒有 transformer 或預訓練 checkpoint。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 4242
rng = np.random.default_rng(SEED)
L = 8
MASK = -1
all_states = ((np.arange(2 ** L)[:, None] >> np.arange(L)) & 1).astype(int)
data_states = all_states[all_states.sum(axis=1) % 2 == 0]

def sample_data(n):
    return data_states[rng.integers(0, len(data_states), size=n)]

def mask_forward(x, mask_probability):
    out = np.array(x, copy=True)
    out[rng.random(out.shape) < mask_probability] = MASK
    return out

x0 = sample_data(6)
print('clean parity states:\n', x0)
print('after absorbing corruption:\n', mask_forward(x0, .55))

In [ ]:
def compatible_states(xt):
    observed = xt != MASK
    return data_states[np.all((data_states == xt) | (~observed), axis=1)]

def exact_conditional(xt):
    candidates = compatible_states(np.asarray(xt))
    if len(candidates) == 0:
        raise ValueError('observations are incompatible with the data distribution')
    prob_one = candidates.mean(axis=0)
    return np.stack([1 - prob_one, prob_one], axis=1)

example = np.array([1, 0, 1, MASK, MASK, MASK, MASK, MASK])
print('compatible joint states:', len(compatible_states(example)))
print('per-position p(0), p(1):\n', exact_conditional(example))

In [ ]:
def sample_joint(xt):
    candidates = compatible_states(xt)
    return candidates[rng.integers(len(candidates))].copy()

def sample_factorized(xt):
    out = np.asarray(xt).copy()
    probs = exact_conditional(out)[:, 1]
    hidden = out == MASK
    out[hidden] = (rng.random(hidden.sum()) < probs[hidden]).astype(int)
    return out

def legal(x):
    return int(np.sum(x) % 2 == 0)

mask_counts = np.arange(L + 1)
joint_valid, factorized_valid = [], []
for m in mask_counts:
    j_ok, f_ok = [], []
    for _ in range(800):
        clean = sample_data(1)[0]
        xt = clean.copy()
        if m:
            xt[rng.choice(L, size=m, replace=False)] = MASK
        j_ok.append(legal(sample_joint(xt)))
        f_ok.append(legal(sample_factorized(xt)))
    joint_valid.append(np.mean(j_ok))
    factorized_valid.append(np.mean(f_ok))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mask_counts, joint_valid, 'o-', label='exact joint reverse')
ax.plot(mask_counts, factorized_valid, 'o-', label='factorized marginals')
ax.set(xlabel='number of masked tokens revealed at once', ylabel='valid parity rate', ylim=(0, 1.05),
       title='Exact marginals do not imply an exact joint sample')
ax.legend()
ax.grid(alpha=.25)
plt.show()

In [ ]:
times = np.linspace(0, 1, 11)
batch = np.repeat(sample_data(300)[:, None, :], len(times), axis=1)
masked_fraction = []
for i, t in enumerate(times):
    corrupted = mask_forward(batch[:, i, :], t)
    masked_fraction.append(np.mean(corrupted == MASK))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(times, masked_fraction, 'o-', label='Monte Carlo')
ax.plot(times, times, '--', label='target schedule $1-\bar\alpha_t=t$')
ax.set(xlabel='t', ylabel='masked fraction', title='Absorbing forward schedule')
ax.legend()
plt.show()

## 讀者練習 / TODO
把長度改成 `L=10`，並把資料改為「1 的數量可被 3 整除」。重建 `data_states`，再畫一次 joint 與 factorized 合法率。哪一種 mask count 最容易暴露高階相依？